In [ ]:
# ─────────────────────────────────────────────────────────────
#  PARAMETERS CELL
# ─────────────────────────────────────────────────────────────
# This is a Fabric "parameters cell". When RunLoadTest launches this notebook
# via runMultiple(), Fabric injects a hidden cell right after this one that
# re-declares every variable passed in the DAG args dict — effectively
# overwriting these defaults. This means:
#   - Standalone run  → these defaults are used as-is
#   - Called by parent → parent values win automatically
#
# You only need to edit this cell if you want to run this notebook by itself.
# ─────────────────────────────────────────────────────────────

# XMLA endpoint of the Power BI workspace hosting the semantic model.
# Set to None to auto-detect the current workspace endpoint.
xmla_endpoint = "powerbi://api.powerbi.com/v1.0/myorg/MRL%20GCTO%20POC"

# Display name of the workspace that hosts the semantic model
workspace = "MRL GCTO POC"

# Path to a Performance Analyzer JSON export inside the lakehouse.
# This file contains the DAX queries that will be replayed.
perf_analyzer_filename = f"/lakehouse/default/Files/PerfScenarios/Queries/{workspace}/PowerBIPerformanceData.json"

# Name of the Power BI semantic model (dataset) to query
model = "Query Import"

# Row-Level Security (RLS) settings:
#   roles            — force specific RLS roles (used together with customdata)
#   customdata       — value accessible via CUSTOMDATA() DAX function in RLS rules
#   effective_username — UPN for RLS impersonation; user must have read+build on the model
roles              = None
customdata         = "foo"
effective_username = None

# How many times to replay the full set of queries from the JSON file
iterations = 1

# Think-time in seconds between consecutive queries (simulates real user pace)
delay_sec = 1

# Unique identifier for this load test run (used in folder names and log rows)
loadtestId = "MyTest"

# This thread's numeric id — used to separate CSV log files in multi-user runs
threadId = 1

# Total number of concurrent threads in this test (written to logs for reference)
concurrent_threads = 1

# Folder where CSV result files are written.  When called from RunLoadTest, the
# parent passes this in.  For standalone runs, it defaults to a standard location.
folder_path = f"/lakehouse/default/Files/PerfScenarios/logs/{workspace}/{loadtestId}"


In [ ]:
# ─────────────────────────────────────────────────────────────
#  HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────
# This cell defines three functions used by Cell 3:
#   run_query()         — sends a single DAX query and counts returned rows
#   load_queries()      — parses a Performance Analyzer JSON into query dicts
#   run_perf_scenario() — replays all queries once, recording timing for each
#
# It also initialises the .NET CLR bridge (via sempy TOM server) so that the
# AdomdConnection class from Microsoft.AnalysisServices is available.
# ─────────────────────────────────────────────────────────────

import json
import time
from typing import List, Dict
from datetime import datetime

import sempy.fabric as fabric

# create_tom_server() loads the .NET CLR into the Python process.
# After this call we can import .NET types like AdomdConnection.
tom = fabric.create_tom_server()

from Microsoft.AnalysisServices.AdomdClient import AdomdConnection


def run_query(con: AdomdConnection, query: str) -> int:
    """
    Execute a single DAX query over an open ADOMD connection.

    Returns the number of rows in the result set.  The result data itself
    is not stored — we only care about row count and elapsed time.
    """
    cmd = con.CreateCommand()
    cmd.CommandText = query
    rows = 0
    rdr = cmd.ExecuteReader()
    while rdr.Read():
        rows += 1
    rdr.Close()
    return rows


def load_queries(fn: str) -> List[Dict[str, str]]:
    """
    Parse a Power BI Performance Analyzer JSON export.

    The JSON contains an array of events.  We pair each
    "Visual Container Lifecycle" event (which carries the visual title/id)
    with the immediately following "Execute DAX Query" event (which carries
    the actual DAX text).  Returns a list of dicts:
        [{"visual_name": ..., "visual_id": ..., "query_text": ...}, ...]
    """
    queries = []
    with open(fn, encoding="utf-8-sig") as f:
        data = json.loads(f.read())

    visual_name = ""
    visual_id = ""
    for event in data["events"]:
        if event["name"] == "Visual Container Lifecycle":
            visual_name = event["metrics"]["visualTitle"]
            visual_id = event["metrics"]["visualId"]
        elif event["name"] == "Execute DAX Query":
            queries.append({
                "visual_name": visual_name,
                "visual_id":   visual_id,
                "query_text":  event["metrics"]["QueryText"],
            })
    return queries


def run_perf_scenario(con: AdomdConnection, queries: List[Dict[str, str]], iteration: int) -> List[dict]:
    """
    Replay every query in *queries* once, measuring wall-clock duration.

    Each result dict captures the full context (model name, thread id,
    iteration number, etc.) so the CSV log is self-describing.
    A configurable think-time (delay_sec) is inserted after each query
    to simulate realistic user pacing.
    """
    results = []
    for qn, q in enumerate(queries, start=1):
        start = time.time()
        rows = run_query(con, q["query_text"])
        duration = time.time() - start

        results.append({
            "loadtest_id":        loadtestId,
            "model":              model,
            "concurrent_threads": concurrent_threads,
            "iterations":         iterations,
            "delay_sec":          delay_sec,
            "query_number":       qn,
            "visual_name":        q["visual_name"],
            "visual_id":          q["visual_id"],
            "iteration":          iteration,
            "query":              q["query_text"],
            "rows":               rows,
            "duration":           duration,
            "start_time":         start,
            "start_time_dt":      datetime.fromtimestamp(start),
            "customdata":         customdata,
            "effective_username": effective_username,
            "thread_id":          threadId,
        })
        time.sleep(delay_sec)
    return results


In [ ]:
# ─────────────────────────────────────────────────────────────
#  EXECUTE QUERIES & WRITE RESULTS
# ─────────────────────────────────────────────────────────────
# This cell:
#   1. Obtains a Power BI access token (used as the XMLA password).
#   2. Builds an ADOMD connection string, optionally adding RLS options.
#   3. Opens the connection and replays all queries for the configured
#      number of iterations.
#   4. Writes the results to a per-user CSV file inside the shared
#      log folder so Evaluation.ipynb can read them later.
#   5. On failure, appends the error to an error.log in the same folder
#      and re-raises so the parent notebook sees the failure.
# ─────────────────────────────────────────────────────────────

import csv

# Obtain a short-lived Power BI access token for the current user.
# This token is used as the password in the XMLA connection string.
token = notebookutils.credentials.getToken("pbi")

# Use folder_path from the parent notebook if provided; otherwise derive it.
if not folder_path:
    folder_path = f"/lakehouse/default/Files/PerfScenarios/logs/{workspace}/{loadtestId}"
notebookutils.fs.mkdirs(folder_path)

print(f"Looking in folder : {folder_path}")
print(f"Files found       : {notebookutils.fs.ls(folder_path)}")

# ── Build the XMLA / ADOMD connection string ──
# Required parts: Data Source (endpoint), Initial Catalog (model), password (token).
# Optional parts added when RLS impersonation or custom data is needed.
constr = f"Data Source={xmla_endpoint};Initial Catalog={model};password={token};Timeout=7200;"
if effective_username is not None:
    constr += f"EffectiveUserName={effective_username};"
if customdata is not None:
    constr += f"CustomData={customdata};"
if roles is not None:
    constr += f"Roles={roles};"

print(f"Connection string: {constr}")

# ── Open connection, run queries, write CSV ──
con = AdomdConnection(constr)
try:
    con.Open()
    queries = load_queries(perf_analyzer_filename)

    all_results = []
    print(f"Starting {iterations} iterations with {delay_sec}s think time between")

    for i in range(iterations):
        results = run_perf_scenario(con, queries, i)
        all_results += results
        # Sleep between iterations (but not after the last one)
        if i < iterations - 1:
            print(f"Completed iteration {i}, sleeping {delay_sec}s")
            time.sleep(delay_sec)

    # Derive a short username for the log filename (e.g. "tartau" from "tartau@merck.com").
    # Falls back to the current notebook user if no impersonation is configured.
    if effective_username:
        username = effective_username.split("@")[0]
    else:
        username = notebookutils.runtime.context["userName"].split("@")[0]

    # Write all results as a CSV.  Filename includes the test id, username,
    # and thread id so multiple users' logs never collide.
    filename = f"{loadtestId}_{username}_{threadId}"

    with open(f"{folder_path}/{filename}.csv", "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=all_results[0].keys())
        writer.writeheader()
        writer.writerows(all_results)

    print(f"Results written to {folder_path}/{filename}.csv")
    con.Close()

except Exception as e:
    # Log the error for post-mortem inspection and re-raise so the parent
    # notebook (RunLoadTest) can detect the failure.
    print(e)
    with open(f"{folder_path}/error.log", "a") as file:
        file.write(f"{constr}  : {e}\n")
    raise
